In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'HOUR/DATE',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

#### Get the data

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()
df = table.to_pandas()
df_copy = table.to_pandas()

#### Separate Date and hour and rename the columns to their shorter counterpart

In [ ]:
df_copy = df_copy.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df_copy['HOUR/DATE'] = pd.to_datetime(df_copy['HOUR/DATE'])

df_copy['DATE'] = df_copy['HOUR/DATE'].dt.date
df_copy['HOUR'] = df_copy['HOUR/DATE'].dt.hour

df_copy = df_copy.drop(columns=['HOUR/DATE'])

#### Compute the gradient

In [ ]:
# Calculate the differential for each identifier separately
df_copy['FLOW'] = df_copy.groupby('POLICY')['CONSUMPTION'].diff()
df_copy = df_copy.dropna(subset=["FLOW"])

#### Make each column to be numerical so that we can compute the correlation matrix

In [ ]:
label_encoders = {}
for col in df_copy.select_dtypes(include='object').columns:  # 'object' dtype selects categorical columns
    le = LabelEncoder()
    df_copy[col] = le.fit_transform(df_copy[col])
    label_encoders[col] = le

#### Plot the correlation matrix

In [ ]:
correlation_matrix = df_copy.corr()
# Display the correlation matrix
plt.figure(figsize=(12, 8))
# Draw a heatmap with square proportional to correlation
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f",
            square=True, linewidths=.5, annot_kws={"size": 8}, cbar_kws={"shrink": .8})
plt.title("Correlation Matrix")
plt.show()